# UNIX and Linux

```{epigraph}
The time has come,' the Walrus said,
      To talk of many things:
Of shoes — and ships — and sealing-wax —
      Of cabbages — and kings —
And why the sea is boiling hot —
      And whether pigs have wings.'

-- Lewis Carroll

```



```{admonition} Learning goals
**By the end of this chapter you should be able to:**
- Navigate the UNIX filesystem using commands like `cd`, `ls`, `pwd`
- Understand the hierarchical directory structure
- Manage files and directories (create, copy, move, delete)
- Use text editors to create and modify files
- Understand and use file permissions
- Use shell features like tab completion, wildcards, and command history
- Search for files and patterns using `find` and `grep`
- Redirect input/output and use pipes to combine commands
```

```{admonition} Prerequisites
:class: tip
**Before starting this chapter, you should:**
- Have access to a UNIX/Linux system or terminal
- Be comfortable using a keyboard for commands (rather than GUI interfaces)
- Have basic computer literacy skills
```

## Introduction

Here we are going to learn to work in a UNIX-like environment, with examples from Linux and macOS. Where administration differs, our Linux examples focus on Ubuntu, a Debian-based distribution. UNIX-style tools are widely used in scientific computing: they let us organise data, combine small programs and automate repetitive tasks. They are not the only way to do serious computing or genomics! Windows users can follow the shell examples in an appropriate Linux environment such as WSL; PowerShell is a different shell with different syntax.

### What is UNIX?

UNIX began at Bell Labs in 1969, led by Ken Thompson and Dennis Ritchie. Its later implementation largely in C helped make it portable across different machines. Its multi-user design, hierarchical filesystem and tools that can exchange plain text make it a useful environment for developing code and handling data. UNIX-like systems can also store and process binary and proprietary file formats; plain text is a useful convention, not a restriction.

Linux is an independently developed UNIX-like kernel, not a direct descendant of the original UNIX source. A Linux distribution such as Ubuntu combines that kernel with command-line tools, libraries and applications. macOS has UNIX foundations through Darwin and BSD, but macOS as a whole is not open-source.

### Why UNIX?

* Small tools can be combined to explore and transform data.
* Shell scripts make repetitive tasks easier to automate and reproduce.
* Multi-user and multitasking systems support shared machines and concurrent work.
* Network tools help us work on remote computers; Linux is especially common on scientific computing clusters.
* Linux and many command-line tools are free and open-source. That does not describe every UNIX system or every application installed on one.
* There is a large body of documentation, tutorials and community support.
* These skills transfer between local work, servers and high-performance computing systems.

```{warning}
UNIX-like systems are not immune to malware or intrusion. Security depends on updates, permissions, configuration and what you choose to run. An open-source licence permits inspection and reuse; it does not by itself guarantee safety.
```

```{note}
**32-bit vs 64-bit:** A 32-bit address space can address $2^{32}$ bytes (4 GiB). A 64-bit address space is much larger, but real memory limits depend on the processor, operating system and hardware. When installing software, match both the operating system and CPU architecture, for example x86-64 or ARM64; "64-bit" alone does not identify a compatible package.
```

## UNIX directory structure

Some key directories on a typical Linux system are:

| Directory | Description |
| :--- | :--- |
| `/` | The root of the filesystem tree. |
| `/bin` | Basic programs; on many Linux systems this is a link to `/usr/bin`. |
| `/etc` | System configuration files. |
| `/dev` | Device files. |
| `/home` | Usually contains users' home directories on Linux, such as `/home/yourname`. |
| `/tmp` | Temporary files; do not keep important work here. |

Your own home directory is not `/home` itself. It is often `/Users/yourname` on macOS, and may be elsewhere on a managed system. Use `echo "$HOME"` to see its path, or `cd` to go there. The shorthand `~` refers to your home directory when expanded by the shell.

This hierarchical structure lets you describe locations with paths, both at the command line and in programs.


---
:::{figure-md} UNIX-directory-structure

```{image} ./graphics/unix-tree.png
:alt: UNIX directory structure
:width: 400px
:class: bg-primary
```

**UNIX directory structure.** The UNIX directory (same as "Folder") structure is hierarchical, with a single tree starting from the "root" `/`. This is quite unlike Windows or MS-DOS, where there are separate trees for disk partitions, removable media, network, etc. <br> <small>(Source: [https://pathanruet.files.wordpress.com/2012/05/unix-tree.png](https://pathanruet.files.wordpress.com/2012/05/unix-tree.png))</small>
:::

---


## Meet the UNIX shell

A **terminal emulator** is the application window in which you type commands and see output. The **shell** is the program that interprets those commands, runs other programs and connects them together. The operating system's **kernel** manages resources such as processes, memory, files and access to hardware. These are different layers, not different names for the same thing.

We will use **Bash**, the Bourne Again Shell, named after the earlier Bourne shell (`sh`).

On Ubuntu's usual desktop, `Ctrl + Alt + T` opens a terminal. On macOS, use `Cmd + Space`, search for Terminal and press Return. Other desktops may use different shortcuts.

```{note}
Recent macOS installations normally start an interactive zsh shell. To follow this chapter's Bash examples consistently, type `bash` in your terminal. This starts a Bash session without changing your login-shell configuration; type `exit` when finished. `echo "$BASH_VERSION"` reports the version in Bash. `$SHELL` usually describes your configured login shell, not necessarily the shell currently interpreting your commands.
```

Let's look at the command prompt:

* A new terminal often starts in your home directory, but an editor's terminal may start in a project directory. Use `pwd` to check.
* A prompt such as `yourname@computer:~$` shows the user, computer and current directory. Its appearance is configurable, so yours may differ.
* The prompt is not part of a command: type only the command that follows it.

```{tip}
`cd` and `cd ~` take you home. `cd ~/Documents` and `cd "$HOME/Documents"` refer to the Documents directory in your home, if that directory exists. Do not type a literal `/home/yourname` path unless that is actually your account's path.
```

Useful shortcuts in Bash's usual Emacs-style line-editing mode include:

| Shortcut | Action |
| :--- | :--- |
| `Ctrl + A` | Go to the beginning of the line. |
| `Ctrl + E` | Go to the end of the line. |
| `Ctrl + L` | Clear/redraw the screen. |
| `Ctrl + U` | Remove text before the cursor. |
| `Ctrl + K` | Remove text after the cursor. |
| `Ctrl + C` | Request interruption of the foreground command, or cancel the input line. |
| `Ctrl + D` | Signal end of input; at an empty prompt this normally exits the shell. |
| `Alt + F` / `Alt + B` | Move forward/backward a word, if the terminal sends these key combinations. |

Use `Tab` to complete command names and paths; when completion is ambiguous, pressing it again often lists alternatives. Try typing `ls /et` and pressing `Tab` before Return. Up/down arrows recall earlier commands. Bindings can differ with terminal settings and line-editing mode.

## Special Characters

The shell interprets some characters before it runs a command. Their meaning depends on context and quoting; we will practise these gradually.

| Character | Meaning |
| :--- | :--- |
| `~` | Home-directory shorthand at the start of an unquoted path. |
| `/` | Separates path components. |
| `#` | Starts a comment when unquoted at the beginning of a shell word. |
| `$` | Introduces expansions, such as `$HOME`, `$(date)` or `$((1 + 2))`. |
| `{` and `}` | Used in several constructs, including brace expansion and command groups. |
| `*` | Filename wildcard matching zero or more characters. |
| `?` | Filename wildcard matching one character. |
| `\` | Escapes the next character outside single quotes; a backslash-newline continues a command. |
| `;` | Separates commands, running the next even if the previous one failed. |
| `&&` | Runs the next command only if the previous one succeeded. |
| `\|` | Pipes standard output from one command to standard input of the next. |
| `>` | Redirects standard output, creating or truncating the target file. |
| `!` | Negates a pipeline's exit status; interactive Bash also uses it for history expansion. |

Quote paths containing spaces, for example `ls "field notes.txt"`. Single quotes preserve literal text; double quotes preserve spaces while still allowing expansions such as `$HOME`. Filename wildcard rules and regular-expression rules are different.

## Basic UNIX commands

Here are some basic commands, many of which you will use frequently. Uppercase names below are placeholders for your own paths or values.

| Command | What it does |
| :--- | :--- |
| `man COMMAND` | Show a command's manual page. |
| `whoami` | Display your user name. |
| `pwd` | Show the current directory. |
| `ls` | List directory entries; hidden names are omitted by default. |
| `cd DIRNAME` | Change directory. |
| `cd ..` | Move one directory up. |
| `cd /` | Go to the root directory. |
| `cd ~` or `cd` | Go to your home directory. |
| `cp FROM TO` | Copy a file. Copying a directory requires a recursive option. |
| `cp -R FROM TO` | Copy a directory and its contents recursively. |
| `mv FROM TO` | Move or rename a file or directory. |
| `touch FILENAME` | Update timestamps; create an empty file if it does not exist. |
| `echo "My string"` | Print a simple string. |
| `rm FILENAME` | Remove a file, not an ordinary directory. |
| `rmdir DIRNAME` | Remove an empty directory. |
| `rm -r DIRNAME` | Remove a directory and its contents recursively. |
| `wc FILENAME` | Count newline characters, words and bytes. `wc -l` counts newlines. |
| `sort FILENAME` | Sort lines and print the result. |
| `uniq FILENAME` | Collapse adjacent repeated lines; separated duplicates remain. |
| `cat FILENAME` | Print a file's contents. |
| `less FILENAME` | Browse a file; press `q` to exit. |
| `head FILENAME` | Print the first ten lines by default. |
| `tail FILENAME` | Print the last ten lines by default. |
| `history` | Show the shell's recorded command history. |
| `date` | Print the current date and time. |
| `file FILENAME` | Identify a file's type from its contents and other information. |
| `passwd` | Change your password, subject to the system's account policy. |
| `chmod MODE FILENAME` | Change permissions; for example, `chmod u+x script.sh`. |

```{warning}
`rm` does not normally move files to a desktop Trash folder. We will practise deletion only on files created for the exercise. Check `pwd` and the target before running a destructive command.
```

```{tip}
`$()`, `$(( ))`, and `${}` are all Bash expansions, used for different purposes!
`${}` is a parameter expansion, used when referencing a variable to make it clear where the variable name ends.
`$()` is a command substitution, which runs a command and inserts its output.
`$(( ))` is an arithmetic expansion, used to evaluate mathematical expressions.

run the code below to test these commands out! 
```

In [1]:
name="Alice"
echo "Hello ${name}!"

echo "Hello on $(date +%F)"

echo "1 + 2 * 3 = $((1 + 2 * 3))"

Hello Alice!
Hello on 2025-10-21
1 + 2 * 3 = 7


### Exploring files and directories

A fundamental command is `ls`, which lists directory entries. Check where you are with `pwd`, then try `ls` and `ls /`. The latter lists entries in the root directory shown in the [filesystem figure](#UNIX-directory-structure).

By default `ls` omits hidden names, which start with a dot. Compare:

```bash
ls -a
ls -l
ls -al
```

`-a` includes hidden entries; `-l` uses the long format. You can combine these flags. An example long-format entry is:

```text
drwxr-xr-x  9 mhasoba mhasoba 4096 Oct 11 10:45 Desktop
```

From left to right:

* The first character is the type: `d` for directory, `-` for regular file, or `l` for symbolic link (a reference to another path).
* The next three triplets are permissions for the owner, the file's group and everyone else. `rwx` means read, write and execute/search permission; a dash means that permission is absent.
* `9` is the hard-link count, not a count of all items inside the directory. Regular files can have more than one hard link. Directory link counts depend on filesystem conventions.
* The next fields name the owner and the group associated with this entry. Matching names do not make that user `root`; groups allow permissions to be shared among users.
* `4096` is the entry's size in bytes. For a directory this is not the total size of the files below it.
* The remaining fields show the modification time and name.

For a **regular file**, read permits reading its contents, write permits changing them, and execute permits attempting to run it as a program. For a **directory**, read permits listing names, execute (search) permits traversing it and accessing named entries, and write together with search normally permits creating, renaming or deleting entries. Changing a file's contents still depends on that file's permissions. Thus `r-x` on Desktop does not guarantee that every file inside it is read-only. Additional controls, such as ACLs and the sticky bit, may also apply.

We will try a small permissions experiment on a disposable file below.

## Building your coursework directory structure

For CMEE coursework, use a single `CMEECourseWork` directory in a suitable location in your home directory, such as Documents. Within it use `week1`, `week2`, etc., each containing `code`, `data`, `results` and `sandbox`. Keep names lowercase exactly as shown. `sandbox` is for scratch work, not assessed files you need to retain. You will bring coursework under version control in the [Version Control chapter](git.ipynb).

```{warning}
First rehearse the following file operations in a **fresh temporary directory**. The next cell creates one and enters it; everything through the search exercises should be done in this practice tree, not in existing coursework. Keep this terminal session open so its variables remain available. Do not copy real coursework into it or delete an existing coursework sandbox to repeat the lesson.
```

After practising, create the same structure in your chosen permanent coursework location, or use your existing structure. Temporary directories are not suitable for storing submissions.

In [ ]:
practice_root=$(mktemp -d)
cd "$practice_root"
pwd

Check what you have in there with:

```bash
ls
```

The fresh practice directory should be empty.

Now make the directory and `cd` to it

In [2]:
mkdir CMEECourseWork

You will get an error if a directory by that name already exists. 

In [3]:
cd CMEECourseWork

You are now inside the practice `CMEECourseWork`. Commands joined with `&&` run the right-hand command only if the left-hand command succeeds; `;` runs the next command regardless of that status.

Predict the output of these two lines before trying them. `missing-directory` does not exist in this fresh tree:

```bash
cd missing-directory && echo "Reached the next command"
cd missing-directory; echo "Reached the next command"
pwd
```

Both `cd` attempts should fail and leave the working directory unchanged. Which `echo` runs? Why would `cd TARGET && COMMAND` matter before a command that writes files?

Next, create your `week1` subdirectory:

In [4]:
mkdir week1

In [5]:
cd week1

In [6]:
mkdir sandbox code data results

From `CMEECourseWork/week1`, predict what happens when the spelling differs:

```bash
(cd Sandbox)
pwd
```

The parentheses run `cd` in a subshell, so your main shell stays in `week1` even if it succeeds. On a case-sensitive filesystem, `Sandbox` and `sandbox` are different names, so this produces a 'No such file or directory' error. On a case-insensitive filesystem it may succeed. What did you observe?

```{warning}
**Case sensitivity depends on the filesystem, not simply the OS.** Many macOS volumes are case-insensitive while preserving the spelling of names; Linux filesystems are usually case-sensitive. This does not make all macOS commands or shell variables case-insensitive. Match case exactly in commands and paths so your work transfers between systems.
```

Stay in the temporary tree's `CMEECourseWork/week1`. Predict what each command will do, then run it and explain any error:

```bash
pwd
mkdir sandbox
rm sandbox
rmdir sandbox
mkdir sandbox
touch sandbox/disposable.txt
rmdir sandbox
```

The first `mkdir` fails because `sandbox` already exists. Plain `rm` refuses an ordinary directory. `rmdir` removes the empty directory, but its second attempt fails because the recreated directory contains a file. These are expected failures, not reasons to add `sudo` or `-f`.

Inspect `ls sandbox` before the next command. It should contain only the disposable file created above. Now try removing this nonempty practice directory recursively:

In [7]:
rm -r sandbox

Careful with the `-r` option, because starting with the target directory (`sandbox` in this case), it `r`ecursively removes the entire directory tree inside the target directory, files and all!


Now recreate your `sandbox`:

In [8]:
mkdir sandbox

Let's try renaming it. On some case-insensitive filesystems, changing only the spelling's case with `mv` may report that the source and destination are the same. If that happens, rename via a distinct unused name: `mv sandbox rename-demo && mv rename-demo Sandbox`. Apply the same two-step approach when changing it back below.

In [9]:
mv sandbox Sandbox

In [10]:
ls

code  data  results  Sandbox


Let's move back to all small caps `sandbox` for naming consistency:

In [11]:
mv Sandbox sandbox
ls

code  data  results  sandbox


Next try some more directory and file manipulations:

In [12]:
cd sandbox

Try `pwd` (`p`rint `w`orking `d`irectory) next to see the full path of the current directory. 

Let's continue:

In [13]:
ls # nothing here yet

The `#` above starts a comment because it is unquoted and begins a shell word. The shell ignores the rest of that line. A quoted `'#'`, or a hash inside a word such as `sample#1`, is literal text instead.

In [14]:
touch TestFile # Or, "touch TestFile.txt"

In [15]:
ls

TestFile


In [16]:
mv TestFile TestFile2; ls # commands can also be sequentially executed with a `;`

TestFile2


In [17]:
rm TestFile2

### Editing a file and changing permissions

You should still be in the practice `CMEECourseWork/week1/sandbox`. Create `notes.txt` with a plain-text editor (for example `nano notes.txt`, if installed), enter a sentence and save it. In nano, `Ctrl + O` then Return saves, and `Ctrl + X` exits. You can instead use your graphical editor to open this exact directory and save the same filename.

```bash
pwd
cat notes.txt
ls -l notes.txt
chmod u-w notes.txt
ls -l notes.txt
chmod u+w notes.txt
ls -l notes.txt
```

Predict which permission character changes. `u` selects the owner; `-w` removes write permission and `+w` restores it. This does not prevent deletion by someone with the appropriate directory permissions. Edit and save a second sentence after restoring write permission.

### Returning to the project root

From `CMEECourseWork/week1/sandbox`, `cd ../..` takes you to `CMEECourseWork`. A second option uses the absolute path saved when we created the temporary directory:

```bash
cd "$practice_root/CMEECourseWork"
pwd
mkdir -p week1/{data,code,results,sandbox}
```

Bash expands the braces into four paths. `mkdir -p` creates missing parents and accepts directories that already exist. Compare that with the earlier plain `mkdir` error. Use exactly the same lowercase spelling throughout. Remain at this practice project root for the next section.

## Command arguments

Most commands accept arguments. Options such as `-l` modify behaviour; operands such as paths specify what to act on. Options are command-specific: the same letter can mean different things to different commands.

| Command | What it does |
| :--- | :--- |
| `cp -R DIR1 DIR2` | Copy recursively, including files and subdirectories. |
| `rm -i FILENAME` | Ask before removing a file. |
| `rm -r DIR` | Remove a directory and its contents recursively. |
| `ls -a` | Include hidden entries. |
| `ls -lh` | Long format with human-readable sizes, such as K, M and G. |
| `ls -l` | Long format. |
| `ls -S` | Order by size, largest first. |
| `ls -t` | Order by modification time, newest first. |
| `ls -1` | One entry per line (the option is the digit one). |
| `ls -r` | Reverse the sorting order. |
| `ls -R` | List subdirectories recursively. |
| `mkdir -p DIR1/DIR2/DIR3` | Create missing directories along this path. |
| `sort -n` | Sort using numeric values: 2 precedes 11. |

You should be at the practice `CMEECourseWork` root. You can combine options; compare the following commands:

In [ ]:
pwd

In [19]:
ls -r

.:
week1

./week1:
code  data  results  sandbox

./week1/code:

./week1/data:

./week1/results:

./week1/sandbox:


`-r` reverses the listing order; it does **not** descend into subdirectories. Compare with `ls -R`, whose uppercase `R` means recursive. Options are case-sensitive.

In [20]:
ls -lr # same as ls -l -r

.:
total 4
drwxrwxr-x 6 mhasoba mhasoba 4096 Oct  5 12:00 week1

./week1:
total 16
drwxrwxr-x 2 mhasoba mhasoba 4096 Oct  5 12:00 code
drwxrwxr-x 2 mhasoba mhasoba 4096 Oct  5 12:00 data
drwxrwxr-x 2 mhasoba mhasoba 4096 Oct  5 12:00 results
drwxrwxr-x 2 mhasoba mhasoba 4096 Oct  5 12:00 sandbox

./week1/code:
total 0

./week1/data:
total 0

./week1/results:
total 0

./week1/sandbox:
total 0


This combines long format (`-l`) with reverse order (`-r`). It shows permissions and modification times, but does not list directory contents recursively. What would `ls -lR` change?

## redirection and pipes


Output of programs can also be "redirected" to a file in two ways:

* `>`: redirect output from a command to a file on disk. If the file already exists, it will be overwritten.

* `>>`: Append the output from a command to a file on disk. If the file does not exist, it will be created.

Let's try some examples.

First make sure you are back in `sandbox`:

In [21]:
cd week1/sandbox

In [22]:
echo "My first line." > test.txt
cat test.txt

My first line.


In [23]:
echo "My second line" >> test.txt
cat test.txt

My first line.
My second line


Now try 

```bash
ls / >> ListrootDir.txt
cat ListrootDir.txt
```
*What do you see?* Cool!

A pipe, `|`, connects the standard output of one command to the standard input of another. For example, `ls / | wc -l` counts newline characters in the listing, usually one per visible entry here. It is not a general file counter: entries can be directories, hidden names are omitted, and names can themselves contain newlines. Error messages normally go to a separate stream, standard error, and are not sent through this pipe.

```bash 
ls / | wc -l
```

*How many is that?* 

Look up `man wc` in the terminal, or search "unix wc" online.

Or try:

In [24]:
ls -lt | head -5

total 4
-rw-rw-r-- 1 mhasoba mhasoba 30 Oct  5 12:01 test.txt


*What does this do?*

## Wildcards

We can use wildcards to match filenames. You should still be in the practice `week1/sandbox`; the next commands create and enter a new `TestWild` directory. These fixtures are empty files because here we are investigating **names**, not contents or sizes.

In [25]:
mkdir TestWild
cd TestWild
touch File1.txt
touch File2.txt
touch File3.txt
touch File4.txt
touch File1.csv
touch File2.csv
touch File3.csv
touch File4.csv
touch Anotherfile.csv
touch Anotherfile.txt
ls 
ls | wc -l

Anotherfile.csv  File1.csv  File2.csv  File3.csv  File4.csv
Anotherfile.txt  File1.txt  File2.txt  File3.txt  File4.txt
10


We will use the following wildcards:

|   Wildcard      | Meaning            |
| :------------- |:-------------| 
|`?`|Any single character, except a leading dot (hidden files).|
|`*`|Zero or more characters, except a leading dot (hidden files).|
|`[A-Z]`|Define a class of characters (e.g., upper-case letters).|                   

Now let's try to find the files using wildcards:

In [26]:
ls *
ls File*
ls *.txt
ls File?.txt
ls File[1-2].txt
ls File[!3].*

Anotherfile.csv  File1.csv  File2.csv  File3.csv  File4.csv
Anotherfile.txt  File1.txt  File2.txt  File3.txt  File4.txt
File1.csv  File2.csv  File3.csv  File4.csv
File1.txt  File2.txt  File3.txt  File4.txt
Anotherfile.txt  File1.txt  File2.txt  File3.txt  File4.txt
File1.txt  File2.txt  File3.txt  File4.txt
File1.txt  File2.txt
File1.csv  File1.txt  File2.csv  File2.txt  File4.csv  File4.txt


(Using-grep)=
## Using `grep`

`grep` searches text and normally prints lines matching a pattern. Its patterns use regular expressions, which we will explore further in the [Python chapter](python_II:Python-regex). Here we will use the supplied protected-species list.

From `TestWild`, the next cell returns to `sandbox` and downloads the file. If `wget` is unavailable (as on many macOS installations), run the `cd` command and use this alternative there:

```bash
curl -fL -o spawannxs.txt https://raw.githubusercontent.com/MulQuaBio/MQB/main/content/data/spawannxs.txt
```

Alternatively, copy the repository's `content/data/spawannxs.txt` into this practice sandbox with your file manager. Choose one method, check that the file is present, then continue with the move to `../data` below.

In [27]:
cd ../ # move one level up from TestWild, back to sandbox
wget https://raw.githubusercontent.com/MulQuaBio/MQB/main/content/data/spawannxs.txt #Cool!

--2021-10-05 12:01:21--  https://raw.githubusercontent.com/MulQuaBio/MQB/main/content/data/spawannxs.txt
resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 13467 (13K) [text/plain]
Saving to: ‘spawannxs.txt’

spawannxs.txt       100%[===================>]  13.15K  --.-KB/s    in 0.003s  

2021-10-05 12:01:21 (5.03 MB/s) - ‘spawannxs.txt’ saved [13467/13467]



In [28]:
ls

spawannxs.txt  test.txt  TestWild


In [29]:
mv spawannxs.txt ../data # Move the file to the appropriate location (note the ../ - relative path)

In [30]:
cd ../data # Move to the data directory

In [31]:
head -n 50 spawannxs.txt #You will see "head" in r and python as well

ANNEX I
List of Species of Marine and Coastal Flora 
Protected Under Article 11(1)(a)

TrACHAEOPHYTA (Vascular Plants)

Family		Genus		Species

Aquifoliaceae	Ilex		cookii
Bignoniaceae	Crescentia	mirabilis
Bignoniaceae	Crescentia	portoricensis
Boraginaceae	Cordia		wagnerorum
Buxaceae	Buxus		vahlii
Cactaceae	Echinocereus 	reichenbachii var. albertii
Cactaceae	Harrisia	fragrans
Cactaceae	Harrisia	portoricensis
Cactaceae	Leptocereus	grantianus
Cactaceae	Leptocereus	wrightii
Cactaceae	Melocactus	guitartii
Cactaceae	Melocactus 	harlowii sensu lato
Cactaceae	Pilosocereus	deeringii
Cactaceae	Pilosocereus	robinii
Convolvulaceae	Bonamia		grandiflora
Convolvulaceae	Ipomoea		flavopurpurea
Convolvulaceae	Ipomoea		walpersiana
Cyatheaceae	Cyathea		dryopteroides
Cyperaceae	rhynchospora	bucherorum
Dioscoreaceae	rajania		theresensis
Ericaceae	rhododendron	chapmanii
Euphorbiaceae	Andrachne	brittonii
Euphorbiaceae	Bernardia	venosa
Euphorbiaceae	Cnidoscolus	fragrans
Euphorbiaceae	Drypetes	triplinervia
Flac

OK, let's look for falcons:

In [32]:
grep Falco spawannxs.txt

Falconidae	Falco		femoralis septentrionalis
Falconidae	Falco		peregrinus
Falconidae	Polyborus	plancus
Falconidae	Falco		columbarius


Nothing there? Let's use `-i` to make the matching case-insensitive:

In [33]:
grep -i Falco spawannxs.txt

Order:	FALCONIFOrMES
Falconidae	Falco		femoralis septentrionalis
Falconidae	Falco		peregrinus
Falconidae	Polyborus	plancus
Order:	FALCONIFOrMES
Order:	FALCONIFOrMES
Order:	FALCONIFOrMES
Falconidae	Falco		columbarius


Now let's find the beautiful ["Ara" macaws](https://en.wikipedia.org/wiki/Ara_(genus)):

In [34]:
grep -i ara spawannxs.txt

Flacourtiaceae	Banaras		vanderbiltii
Order:	CHArADrIIFOrMES
Charadriidae	Charadrius	melodus
Psittacidae	Amazona		arausica
Psittacidae	Ara		macao
Dasyproctidae	Dasyprocta	guamara
Palmae		Syagrus (= rhyticocos)	amara
Psittacidae	Ara		ararauna
Psittacidae	Ara		chloroptera
Psittacidae	Arao		manilata
Mustelidae	Eira		barbara
Order:	CHArADrIIFOrMES


But this poses a problem (*What is the problem?*). 

We can solve this by specifying `-w` to match only full words:

In [35]:
grep -i -w ara spawannxs.txt

Psittacidae	Ara		macao
Psittacidae	Ara		ararauna
Psittacidae	Ara		chloroptera


And also show line(s) after the one that was matched, we can use `-A x`, where `x` is number of lines to use:

In [36]:
grep -i -w -A 1 ara spawannxs.txt

Psittacidae	Ara		macao

--
Psittacidae	Ara		ararauna
Psittacidae	Ara		chloroptera
Psittacidae	Arao		manilata


Similarly, <span>-B</span> shows the lines before:

In [37]:
grep -i -w -B 1 ara spawannxs.txt

Psittacidae	Amazona		vittata
Psittacidae	Ara		macao
--
Psittacidae	Amazona		ochrocephala
Psittacidae	Ara		ararauna
Psittacidae	Ara		chloroptera


Use <span>-n</span> to show the line number of the match:

In [38]:
grep -i -w -n ara spawannxs.txt

216:Psittacidae	Ara		macao
461:Psittacidae	Ara		ararauna
462:Psittacidae	Ara		chloroptera


To print all the lines that do not match a pattern, use <span>-v</span>:

Also try `grep -i -w -v ara spawannxs.txt`

This will produce a lot of output (which I will not show)! *What does this command do?*

To match either of two patterns, use extended regular expressions with straight quotes and a space after the command:

```bash
grep -i -E 'falco|ara' spawannxs.txt
```

The quotes keep the shell from interpreting `|` as a pipe; here it means alternation to `grep -E`. Compare with `grep -i -w -E 'falco|ara' spawannxs.txt`. Does matching a substring answer the same question as matching a whole word? `grep` prints matching **lines**, even when a line contains several matches. A status of 1 means no lines matched, not a syntax error.

## Finding files

`find` walks a directory tree and tests entries. You should currently be in the practice `week1/data`. The next command takes you to `week1/sandbox`; check with `pwd` before creating `TestFind`.

In [39]:
cd ../sandbox # again, note the relative path!

In [40]:
mkdir TestFind
cd TestFind
mkdir -p Dir1/Dir11/Dir111 #what does -p do?
mkdir Dir2
mkdir Dir3
touch Dir1/File1.txt
touch Dir1/File1.csv
touch Dir1/File1.tex
touch Dir2/File2.txt
touch Dir2/file2.csv
touch Dir2/File2.tex
touch Dir1/Dir11/Dir111/File111.txt
touch Dir3/File3.txt

In [41]:
ls

Dir1  Dir2  Dir3


In [42]:
ls Dir1

Dir11  File1.csv  File1.tex  File1.txt


In [43]:
ls Dir* # the asterisk wildcards names so you can find all dirs with a particular start to their name

Dir1:
Dir11  File1.csv  File1.tex  File1.txt

Dir2:
file2.csv  File2.tex  File2.txt

Dir3:
File3.txt


Don't forget to use `man ` to check what `mkdir` and `touch` do. 

Now find particular files:

In [44]:
find . -name "File1.txt"

./Dir1/File1.txt


Using `-iname` ignores case, and you can use wildcards:

In [45]:
find . -iname "fi*.txt"

./Dir1/File1.txt
./Dir1/Dir11/Dir111/File111.txt
./Dir3/File3.txt
./Dir2/File2.txt


You can limit how deeply `find` descends. The starting point `.` has depth 0, its immediate entries depth 1, and entries inside those directories depth 2. Thus `-maxdepth 2` includes files in `Dir1`, `Dir2` and `Dir3`, but not the deeper `File111.txt`. `-maxdepth 1` would restrict the search to immediate entries. `-maxdepth` is available in GNU find and current macOS find, but is not a POSIX option; check `man find` on other systems.

In [46]:
find . -maxdepth 2 -name "*.txt"

./Dir1/File1.txt
./Dir3/File3.txt
./Dir2/File2.txt


Negating a name test excludes matching names. Predict the following output: without `-type f`, directories (including `.`) can appear as well as files. How would adding `-type f` change the result?

In [47]:
find . -maxdepth 2 -not -name "*.txt"

.
./Dir1
./Dir1/File1.tex
./Dir1/File1.csv
./Dir1/Dir11
./Dir3
./Dir2
./Dir2/File2.tex
./Dir2/file2.csv


To find only directories whose names contain `dir`, ignoring case, use `-type d` with `-iname`. Quote the pattern so **find**, not the shell, interprets the wildcard. The same quoting rule applies to the earlier `-name` examples:

In [ ]:
find . -type d -iname '*dir*'

./Dir1
./Dir1/Dir11
./Dir1/Dir11/Dir111
./Dir3
./Dir2


## Installing or removing software in Ubuntu

You can install software in your `/home` directory. In UNIX you originally had to login as `root` (administrator). But in Ubuntu, it is sufficient to add `sudo` (`s`uper `u`ser `do`) in front of a command. To install packages in all Debian (including Ubuntu) distributions you need to use the `apt` group of commands which allow you to deal with various installation (and un-installation) tasks.

---

:::{figure-md} UNIX-sudo

```{image} ./graphics/sudo.png
:alt: sudo
:width: 250px
```

**`sudo` rules.** <small>(Source: [XKCD](http://xkcd.com/149/))</small>

:::

---

### Installing from the main repository

You can install anything that is in the main Ubuntu package/software repository.

&#9733; Try installing r, which is pre-packaged the main Ubuntu repository:

```bash

sudo apt install r-base

```

This will prompt you for your password.  

```{tip}
**Using the `tab` key in terminal**: Don't forget to use the `tab` key in the terminal. It aut-ocompletes directory/file names for you (same in `r`and `python`, and/or provides a list of files in the current directory (hit `tab` twice after certain commands. For example, try double tab after typing `ls`at the bash prompt.)
```

### Enabling additional repositories

You can enable additonal repositories that are already present (but not enabled by default) by using the Ubuntu "Software Centre" GUI. Launch Ubuntu Software Centre, and then Edit > Software Sources > Other Software (the exact approach will vary depnding on your Debian or Ubuntu distribution).

### Installing from other repositories

In addition there are other repositories you can access, but which first need to be auhtenticated locally for your system's security and stability. There are typically available in the form of *Personal Package Archives* (PPAs). The general approach is: 

* Find the repository's web page (it's Launchpad page)
* Look for "Adding this PPA to your system" on the page. Make a note of the PPA's location, which has the format ppa:user/ppa-name.
* Open a terminal and enter: `sudo add-apt-repository ppa:user/ppa-name`. (replace 'ppa:user/ppa-name' with the PPA's location that you noted above)

Your system will now fetch the PPA's key. This enables your system to verify that the packages in the PPA have not been interfered with since they were built.

* Finally, download the latest list of software from each repository archive on your system (all of them: main, additional, and extra ones you have added manually such as the PPA you just added. To do this, use:

```bash
sudo apt-get update
```
You can now start installing software from the new repository. 

&#9733; Try installing something that isn't in the main repository &ndash; a weather indicator. 

For this, you first have to add the repository:

```bash
sudo add-apt-repository ppa:atareao/atareao
```

This will ask you to first approve the addition of this repository. Then, `sudo apt-get update`, and then, finally,

```bash
sudo apt install my-weather-indicator

```

```{tip}

**Ubuntu repositories:** Official repositories are defined in `/etc/apt/sources.list`. You can enable or disable individual repositories by editing that file. 

```

### Installing from downloaded `*.deb` files

You can also install software from third-party sources by downloading an installable package archive. Such archives come with a `*.deb` extension. 

For example, you can install [Visual Studio Code](https://code.visualstudio.com/) this way. Let's try it.

&#9733; First download and install the .deb package (64-bit) from their website (google it!). This will save it to your `Downloads` directory.Then `cd` to its location:

```bash
cd ~/Downloads
```

This is same as typing `cd /home/<YourUserName>/Downloads`. Then, you can install using:

```bash
sudo dpkg -i code_1.71.2-1663191218_amd64.deb # install, including dependencies
```

Installing the `.deb` package will automatically also install the softwarre's apt repository and authorize it. That means henceforth, the program `code` will automatically auto-update during the regular system updates. 

### removing software

You can also easily remove software by, well, using the `remove` command! 

You will find that the list of UNIX commands are quite exhaustive, and their names quite intuitive, as they should be. If you think a certain command with a certain name should exist, it very often does!

&#9733; Try removing the totem video player and installing the more versatile VLC player instead:

```bash 
sudo apt remove totem
```
and then, 

```bash 
sudo apt install vlc

```

## Particularly useful UNIX tools

We have only scratched the surface. Many small command-line tools can be combined for scientific work. Some are already installed; others require platform-appropriate installation.

| Tool | What it does |
| :--- | :--- |
| `top` | Inspect processes and their CPU/memory use. |
| `kill` | Send a signal to a process; check the process ID and use with care. |
| `grep` | Search text for patterns, including text received through a pipe. |
| `find` | Search filesystem entries by name and other properties. |
| `pandoc` | Convert document formats; some output formats require additional tools. |
| ImageMagick | Convert and manipulate images (`magick` in version 7; older examples use `convert`). |
| `pdftk` | Manipulate PDFs, such as merging or splitting documents. |
| `wget` / `curl` | Transfer data, including downloading files from the web. |
| `rsync` | Copy and synchronise files locally or over a network. |
| `cron` | Schedule recurring tasks on systems that provide it. |
| `unison` | Synchronise files between locations. |

Next you will learn to automate tools using [shell scripts](shell-scripting.ipynb). Consult `man COMMAND` or `whatis COMMAND`. Many programs support `--help`, but that option is not universal; shell builtins in Bash can instead be explored with `help`, for example `help cd`.

## Practicals

### Instructions

* Review the commands above and explain their outcomes, including the deliberately failing examples. System-specific output may differ from the stored illustrations.
* Make sure your **permanent** coursework directory (e.g., `CMEECourseWork/week1`) is organised with `code`, `data`, `results` and `sandbox`, and contains the necessary files. The temporary rehearsal tree is not your submission directory.
* Along with completeness of the practicals/exercises themselves, you will be marked on how complete and well-organised your directory structure and content is, in all coming weeks as well.
* Please put judicious comments in all your script files.

### Bash command challenge

Here is a deliberately inefficient Bash pipeline using two pipes:

```bash
find . -type f -exec ls -s {} \; | sort -n | head -10
```

The earlier empty files are useful for testing names, but poor fixtures for investigating sizes. In the **temporary practice tree**, create some different nonempty files:

```bash
cd "$practice_root/CMEECourseWork/week1/sandbox"
mkdir SizeChallenge
cd SizeChallenge
printf 'oak\n' > small.txt
printf 'two field observations\n' > "field notes.txt"
dd if=/dev/zero of=medium.bin bs=1024 count=8
dd if=/dev/zero of=large.bin bs=1024 count=32
ls -ls
```

Run the original pipeline here. Predict the ordering first, and investigate ties. There are only four files, so `head -10` will display all four lines. Keep this exercise inside `SizeChallenge`, not a directory containing valuable data.

Here's what the original pipeline does:

* `find . -type f` searches recursively from `.` for regular files.
* `-exec ls -s {} \;` runs a separate `ls` for each file. `{}` is replaced by the path; the escaped semicolon terminates the `-exec` action, not the whole shell command.
* `ls -s` reports **allocated disk space in blocks**, not the logical length of the file in bytes. GNU ls normally uses 1024-byte blocks; macOS/BSD ls normally uses 512-byte blocks. Environment settings can change units. `ls -sk` requests 1024-byte units on these systems; consult `man ls`. Compare with the byte-size column of `ls -l`. Small files can occupy the same number of blocks even when their contents differ in length; sparse files can have large logical sizes but little allocated space.
* `sort -n` sorts numerically by the leading block count, smallest first.
* `head -10` keeps up to ten lines. Under these controlled filename assumptions, these represent the files with the smallest allocated sizes, not necessarily the shortest contents.

Before optimising, explore batching on a **different task**: displaying the paths of the text files in the earlier `TestWild` fixture.

```bash
find ../TestWild -type f -name '*.txt' -exec printf 'path: %s\n' {} \;
find ../TestWild -type f -name '*.txt' -exec printf 'path: %s\n' {} +
```

Both should display the same paths (traversal order need not be fixed). The first invokes the command for each path; the second passes batches of paths as separate arguments. Batches respect command-line size limits, so `+` does not guarantee exactly one invocation. The path containing spaces in `SizeChallenge` provides a useful check when you apply this idea yourself.

Now do the following, explaining each answer in words:

1. Modify the original pipeline so it prints sizes in human-readable units. Keep the intended size ordering: `sort -n` does not interpret unit suffixes, so compare values such as `900K` and `2M`. Consult `man sort` for human-numeric sorting (`-h` where available), or consider when to format numbers for display.
2. Modify the original pipeline so that `ls -s` handles as many files per invocation as possible, instead of one at a time. Use `man find` and the separate batching example above. Inspect the output when changing invocation shape: `ls` may add headings or a `total` line in some modes. Which lines actually represent files?
3. Find another solution that avoids calling `ls` once per file. Investigate `xargs`. Before relying on your result, check how it handles `field notes.txt`: ordinary whitespace-separated input can split a single filename into several arguments, and quotes/backslashes can also be interpreted.

```{tip}
Optional extension: investigate `find -print0` and `xargs -0` for transporting filenames separated by NUL bytes, including names with spaces or newlines. This protects argument boundaries; it does not automatically make a later line-based `ls | sort | head` report safe for newline-containing names. Our displayed-size exercise uses filenames without newlines. Do not confuse robust filename transport with sorting a human-readable report.
```

### FASTA exercise

The supplied files are in the book repository's `content/data/fasta` directory, not an absolute `/data/fasta` directory on your computer. Copy the `fasta` directory into your permanent coursework `week1/data`, leaving the supplied originals unchanged. Work from `week1/code`, referring to inputs under `../data/fasta/`. The three filenames are `407228326.fasta`, `407228412.fasta` and `e_coli.fasta`.

Each supplied [FASTA file](https://en.wikipedia.org/wiki/FASTA_format) contains **one record**: a header starting with `>`, followed by sequence data wrapped across lines. Removing just the first line is appropriate for these inputs, not for arbitrary multi-record FASTA files. Wrapping is a presentation choice, not part of the biological sequence.

Write `unixPrac1.txt` with single-line UNIX commands, numbering each answer with a comment such as `#1`, `#2`, etc.:

* Count the lines in each file. Interpret `wc -l` as a **newline count**: an unterminated final line is not included in that count. The two numbered FASTA files lack a final newline; explain the distinction from counting displayed text lines.
* Print everything starting from the second line of `e_coli.fasta`.
* Count the sequence length of this genome: exclude its header and line endings, but include ambiguous nucleotide positions.
* Count occurrences of the literal motif `ATGC` in the *E. coli* sequence, not just the number of lines containing it. Remove the header and line breaks first so wrapping does not hide a motif. `grep -c` counts matching lines. Treat case consistently; ambiguity codes are not literal A, T, G or C matches. This particular motif cannot overlap with itself.
* Compute the AT/GC ratio, $(A+T)/(G+C)$, as a single number, using counts of the four canonical DNA bases. This is not the same as the GC fraction, $(G+C)/(A+T+G+C)$. Exclude ambiguous positions from these four counts without guessing their identities.

DNA ambiguity codes are standardised by IUPAC: `R` means A or G, `Y` means C or T, `M` means A or C, `W` means A or T, and `N` means any base; other defined codes cover the remaining combinations. They denote DNA bases here, not ribonucleotides. Keep ambiguous symbols in the sequence used for length and motif calculations: deleting them could join previously separated bases and create a false motif. For example, removing `N` from `ATNGC` would invent `ATGC`.

GC content affects duplex melting behaviour under specified conditions, but thermal stability also depends on sequence, salt and other factors. It is not valid to infer that every thermophile has a GC-rich genome, or to use a genome's AT/GC ratio alone to diagnose its habitat.

```{note}
For interpretation beyond these supplied inputs: when `G+C` is zero, the AT/GC ratio has no finite value; report it as undefined rather than zero. The supplied files have nonzero GC counts. This is a discussion of an edge case, not an additional submission task.
```

* Save `unixPrac1.txt` in your permanent `code` directory.
* Each command must read its input from the `data` directory via the appropriate relative path.

Do **not** turn these answers into shell scripts yet; each answer should be a single-line solution made of potentially piped UNIX commands. Automation comes next in the [Shell Scripting chapter](shell-scripting.ipynb).

## readings & resources

Your University library likely gives you access to several e- and paper books on UNIX, some specific to Ubuntu. Browse or search and find a intro book that suits you.

### General 
* If you like history, [see this]( https://www.howtogeek.com/182649/htg-explains-what-is-unix).

* Lots of UNIX tutorials out there. Try [http://software-carpentry.org/lessons.html](http://software-carpentry.org/lessons.html) (Chapter “shell”).

* The bash reference manual: [https://www.gnu.org/software/bash/manual/bash.html](https://www.gnu.org/software/bash/manual/bash.html)

### repositories and packages in Ubuntu

* read about [Apt authentication](https://help.ubuntu.com/community/SecureApt) in Debian Linux distributions (including Ubuntu).

* [What are repositories in Ubuntu?](https://help.ubuntu.com/community/repositories/Ubuntu)

* [How to use repositories](https://help.ubuntu.com/community/repositories/Ubuntu)
